# LeetCode 1679 — Max Number of K-Sum Pairs bằng Python

Link đề: https://leetcode.com/problems/max-number-of-k-sum-pairs/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Two Pointers, Sorting

Skeleton LeetCode:

```python
class Solution:
    def maxOperations(self, nums: List[int], k: int) -> int:
        ...
```

Notebook đi từ **thử mọi cách ghép cặp** (vét cạn theo định nghĩa) → **tham lam `O(n²)` với mảng đánh dấu** → **sắp xếp + hai con trỏ** → **bảng băm một lượt** → **công thức đếm theo `Counter`**.

## 1. Mục tiêu học

1. Hiểu bài là **ghép cặp rời nhau** (mỗi phần tử dùng tối đa một lần), không phải đếm số cặp chỉ số `i < j`.
2. Thấy rằng chỉ **giá trị** quan trọng: số thao tác tối đa = `Σ min(cnt[x], cnt[k−x])` trên các `x < k − x`, cộng `cnt[k/2] // 2`.
3. Cài đặt **hai con trỏ** trên mảng đã sắp xếp và chứng minh vì sao bỏ một đầu là an toàn.
4. Cài đặt **bảng băm một lượt** "chờ ghép" — mẫu chung của bài Two Sum.
5. Kiểm thử bằng vét cạn ghép cặp trên mảng nhỏ, seed cố định.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` và số nguyên `k`. Mỗi **thao tác**, bạn chọn **hai số** trong mảng có **tổng bằng `k`** và **xóa** chúng khỏi mảng.

Trả về **số thao tác tối đa** có thể thực hiện.

**Ví dụ 1:** `nums = [1,2,3,4]`, `k = 5` → `2`. Xóa `1` và `4` ⇒ `[2,3]`; xóa `2` và `3` ⇒ `[]`.

**Ví dụ 2:** `nums = [3,1,3,4,3]`, `k = 6` → `1`. Xóa hai số `3` đầu ⇒ `[1,4,3]`; không còn cặp nào tổng 6.

**Ràng buộc:**
- `1 ≤ nums.length ≤ 10⁵`
- `1 ≤ nums[i] ≤ 10⁹`
- `1 ≤ k ≤ 10⁹`

In [1]:
import random
import time
from collections import Counter
from functools import lru_cache
from typing import List

examples = [([1, 2, 3, 4], 5, 2), ([3, 1, 3, 4, 3], 6, 1)]

def simulate(nums, k, plan):
    """plan: list các cặp chỉ số (i, j) theo mảng GỐC. Kiểm tra hợp lệ và trả số thao tác."""
    used = set()
    for i, j in plan:
        assert i != j and i not in used and j not in used and nums[i] + nums[j] == k
        used |= {i, j}
    return len(plan)

print("Ví dụ 1, ghép (0,3), (1,2):", simulate([1, 2, 3, 4], 5, [(0, 3), (1, 2)]))
print("Ví dụ 2, ghép (0,2):", simulate([3, 1, 3, 4, 3], 6, [(0, 2)]))

Ví dụ 1, ghép (0,3), (1,2): 2
Ví dụ 2, ghép (0,2): 1


## 3. Kiến thức nền

### 3.1 `Counter` và tra "phần bù" `k − x`

Với mỗi giá trị `x`, phần tử ghép được với nó phải có giá trị đúng `k − x`. `Counter(nums)` cho biết mỗi giá trị xuất hiện bao nhiêu lần.

In [2]:
nums = [3, 1, 3, 4, 3]
c = Counter(nums)
print(c)
for x in sorted(c):
    print(f"x={x}: cần {6 - x}, có {c.get(6 - x, 0)} phần tử như vậy")

Counter({3: 3, 1: 1, 4: 1})
x=1: cần 5, có 0 phần tử như vậy
x=3: cần 3, có 3 phần tử như vậy
x=4: cần 2, có 0 phần tử như vậy


### 3.2 Sắp xếp và hai con trỏ

Trên mảng **tăng dần**, đặt `l` ở đầu, `r` ở cuối: tổng `nums[l] + nums[r]` **tăng** khi `l` tăng và **giảm** khi `r` giảm. Đó là nền tảng của mọi bài "tìm cặp có tổng cho trước" trên mảng sắp xếp.

In [3]:
a = sorted([1, 2, 3, 4, 6, 8])
l, r = 0, len(a) - 1
while l < r:
    print(f"l={l} ({a[l]}), r={r} ({a[r]}), tổng={a[l] + a[r]}")
    if a[l] + a[r] < 9: l += 1
    else: r -= 1

l=0 (1), r=5 (8), tổng=9
l=0 (1), r=4 (6), tổng=7
l=1 (2), r=4 (6), tổng=8
l=2 (3), r=4 (6), tổng=9
l=2 (3), r=3 (4), tổng=7


### 3.3 Đệ quy có nhớ trên `tuple` (để vét cạn)

Biểu diễn trạng thái "mảng còn lại" bằng một `tuple` đã sắp xếp (đa tập), dùng `lru_cache` để không giải lại cùng trạng thái.

In [4]:
@lru_cache(maxsize=None)
def count_states(t):
    return 1 if not t else 1 + count_states(t[1:])

print(count_states((1, 2, 3)), count_states.cache_info().currsize, "trạng thái được nhớ")

4 4 trạng thái được nhớ


### 3.4 Chia nguyên `//` cho trường hợp `x = k − x`

Khi `k` chẵn và `x = k / 2`, hai phần tử **cùng giá trị** ghép với nhau ⇒ số cặp từ `cnt[x]` phần tử là `cnt[x] // 2`.

In [5]:
for cnt in [1, 2, 3, 4, 5]:
    print(f"{cnt} số 3 (k=6) -> {cnt // 2} cặp")

1 số 3 (k=6) -> 0 cặp
2 số 3 (k=6) -> 1 cặp
3 số 3 (k=6) -> 1 cặp
4 số 3 (k=6) -> 2 cặp
5 số 3 (k=6) -> 2 cặp


## 4. Ý tưởng nền tảng / trực giác

Mọi cặp hợp lệ có dạng `{x, k − x}`. Các nhóm giá trị **độc lập**: nhóm `{1, 5}` không ảnh hưởng nhóm `{2, 4}` khi `k = 6`. Trong mỗi nhóm:
- `x ≠ k − x`: số cặp tối đa = `min(cnt[x], cnt[k − x])` (mỗi cặp dùng một `x` và một `k − x`).
- `x = k − x`: số cặp = `cnt[x] // 2`.

Tính tay `nums = [3,1,3,4,3]`, `k = 6`: `cnt = {3: 3, 1: 1, 4: 1}`.
- Nhóm `{1, 5}`: `min(1, 0) = 0`.
- Nhóm `{2, 4}`: `min(0, 1) = 0`.
- Nhóm `{3}` (vì `3 = 6 − 3`): `3 // 2 = 1`.
⇒ tổng `1`.

Tính tay `nums = [1,1,1,5,5,2,4,4,4]`, `k = 6` (ví dụ tự dựng): nhóm `{1, 5}`: `min(3, 2) = 2`; nhóm `{2, 4}`: `min(1, 3) = 1` ⇒ `3`.

In [6]:
def by_groups(nums, k):
    c = Counter(nums)
    total = 0
    for x in sorted(c):
        y = k - x
        if x < y and y in c:
            print(f"  nhóm {{{x}, {y}}}: min({c[x]}, {c[y]}) = {min(c[x], c[y])}")
            total += min(c[x], c[y])
        elif x == y:
            print(f"  nhóm {{{x}}}: {c[x]} // 2 = {c[x] // 2}")
            total += c[x] // 2
    print("  tổng =", total)
    return total

print("[3,1,3,4,3], k=6:"); by_groups([3, 1, 3, 4, 3], 6)
print("[1,1,1,5,5,2,4,4,4], k=6:"); by_groups([1, 1, 1, 5, 5, 2, 4, 4, 4], 6)

[3,1,3,4,3], k=6:
  nhóm {3}: 3 // 2 = 1
  tổng = 1
[1,1,1,5,5,2,4,4,4], k=6:
  nhóm {1, 5}: min(3, 2) = 2
  nhóm {2, 4}: min(1, 3) = 1
  tổng = 3


3

## 5. Cách 1 — Brute Force / baseline

### 5.1 Vét cạn theo định nghĩa: thử mọi cách ghép

Lấy phần tử nhỏ nhất còn lại; hoặc **bỏ** nó (không ghép), hoặc ghép với **một** phần tử có giá trị `k − x`. Đệ quy có nhớ trên đa tập còn lại. Chỉ dùng cho mảng nhỏ.

In [7]:
def max_ops_definition(nums, k):
    @lru_cache(maxsize=None)
    def best(t):
        if len(t) < 2:
            return 0
        x, rest = t[0], t[1:]
        res = best(rest)                              # bỏ x
        y = k - x
        if y in rest:
            i = rest.index(y)
            res = max(res, 1 + best(rest[:i] + rest[i + 1:]))   # ghép x với một y
        return res
    return best(tuple(sorted(nums)))

for nums_, k_, exp in examples:
    print(max_ops_definition(nums_, k_), "(kỳ vọng", exp, ")")

2 (kỳ vọng 2 )
1 (kỳ vọng 1 )


### 5.2 Tham lam `O(n²)` với mảng đánh dấu

Với mỗi `i` chưa dùng, tìm `j > i` chưa dùng có `nums[i] + nums[j] = k`; nếu có thì ghép và đánh dấu cả hai.

In [8]:
def max_ops_quadratic(nums, k):
    n = len(nums)
    used = [False] * n
    ops = 0
    for i in range(n):
        if used[i]:
            continue
        for j in range(i + 1, n):
            if not used[j] and nums[i] + nums[j] == k:
                used[i] = used[j] = True
                ops += 1
                break
    return ops

for nums_, k_, exp in examples:
    print(max_ops_quadratic(nums_, k_), "(kỳ vọng", exp, ")")

2 (kỳ vọng 2 )
1 (kỳ vọng 1 )


**Độ phức tạp:** 5.1 hàm mũ (số đa tập con); 5.2 `O(n²)` thời gian — với `n = 10⁵` là `5·10⁹`: quá chậm. Bộ nhớ `O(n)`.

**Nút thắt:** để tìm "phần bù" `k − x` ta quét tuyến tính. Có thể tra **trực tiếp** bằng bảng băm `O(1)`, hoặc nhờ **thứ tự** (sắp xếp) mà loại bỏ cả loạt ứng viên.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (chỉ giá trị quan trọng):** phần tử `x` chỉ ghép được với phần tử giá trị `k − x`. Ghép tham lam bất kỳ `x` với bất kỳ `k − x` chưa dùng đều **không làm hại** — vì mọi bản sao của `k − x` đều tương đương.

**Insight 2 (hai con trỏ):** trên mảng tăng dần với `l < r`:
- **Nếu** `a[l] + a[r] == k` **thì** ghép, `l += 1`, `r −= 1`.
- **Nếu** `a[l] + a[r] > k` **thì** `a[r]` quá lớn: với **mọi** `l' ≥ l`, `a[l'] + a[r] ≥ a[l] + a[r] > k` ⇒ `a[r]` không ghép được với ai còn lại ⇒ bỏ `r`.
- **Nếu** `< k` **thì** đối xứng: bỏ `l`.

**Insight 3 (bảng băm một lượt):** duyệt `x`; **nếu** có một `k − x` đang "chờ" **thì** ghép (giảm số chờ, tăng đáp án); **ngược lại** cho `x` vào hàng chờ.

**Bất biến (hai con trỏ):** số cặp đã ghép + số cặp tối đa trong `a[l..r]` = đáp án của toàn mảng.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Sắp xếp + hai con trỏ

In [9]:
def max_ops_two_pointers(nums, k):
    a = sorted(nums)
    l, r, ops = 0, len(a) - 1, 0
    while l < r:
        s = a[l] + a[r]
        if s == k:
            ops += 1
            l += 1
            r -= 1
        elif s > k:
            r -= 1
        else:
            l += 1
    return ops

class SolutionTwoPointers:
    def maxOperations(self, nums: List[int], k: int) -> int:
        a = sorted(nums)
        l, r, ops = 0, len(a) - 1, 0
        while l < r:
            s = a[l] + a[r]
            if s == k:
                ops += 1; l += 1; r -= 1
            elif s > k:
                r -= 1
            else:
                l += 1
        return ops

for nums_, k_, exp in examples:
    print(max_ops_two_pointers(nums_, k_), SolutionTwoPointers().maxOperations(nums_, k_), "(kỳ vọng", exp, ")")

2 2 (kỳ vọng 2 )
1 1 (kỳ vọng 1 )


**Độ phức tạp:** `O(n log n)` (sắp xếp) + `O(n)` (quét); bộ nhớ `O(n)` cho bản sao đã sắp (hoặc `O(1)` nếu được sắp tại chỗ).

### 7.2 Cách 3 — Bảng băm một lượt ("chờ ghép")

In [10]:
def max_ops_hash(nums, k):
    waiting = Counter()
    ops = 0
    for x in nums:
        if waiting[k - x] > 0:
            waiting[k - x] -= 1
            ops += 1
        else:
            waiting[x] += 1
    return ops

for nums_, k_, exp in examples:
    print(max_ops_hash(nums_, k_), "(kỳ vọng", exp, ")")

2 (kỳ vọng 2 )
1 (kỳ vọng 1 )


**Độ phức tạp:** `O(n)` thời gian trung bình, `O(n)` bộ nhớ. Chú ý: phải kiểm `waiting[k − x] > 0` **trước** khi thêm `x`. Với `Counter`, đọc một khóa chưa có trả về `0` và **không** chèn khóa mới, nên đọc thoải mái.

### 7.3 Cách 4 — Công thức đếm theo nhóm giá trị

In [11]:
def max_ops_counter(nums, k):
    c = Counter(nums)
    ops = 0
    for x, cx in c.items():
        y = k - x
        if x < y:
            ops += min(cx, c.get(y, 0))
        elif x == y:
            ops += cx // 2
    return ops

for nums_, k_, exp in examples:
    print(max_ops_counter(nums_, k_), "(kỳ vọng", exp, ")")

2 (kỳ vọng 2 )
1 (kỳ vọng 1 )


Chỉ cộng khi `x < y` (hoặc `x == y`) để mỗi nhóm `{x, k − x}` được tính **đúng một lần**. **Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ.

## 8. TRACE từng bước

### 8.1 Trace cách 2 (hai con trỏ) trên `nums = [3,1,3,4,3]`, `k = 6`

Sau sắp xếp: `[1, 3, 3, 3, 4]`.

| l | r | a[l] | a[r] | tổng | hành động | ops |
|---|---|---|---|---|---|---|
| 0 | 4 | 1 | 4 | 5 | < 6 ⇒ l++ | 0 |
| 1 | 4 | 3 | 4 | 7 | > 6 ⇒ r−− | 0 |
| 1 | 3 | 3 | 3 | 6 | ghép | 1 |
| 2 | 2 | — | — | — | dừng (l = r) | 1 |

In [12]:
def trace_two_pointers(nums, k):
    a = sorted(nums)
    print("sắp xếp:", a)
    l, r, ops = 0, len(a) - 1, 0
    while l < r:
        s = a[l] + a[r]
        if s == k:
            ops += 1; act = f"== {k}: ghép ({a[l]}, {a[r]})"
            l += 1; r -= 1
        elif s > k:
            act = f"> {k}: bỏ a[r]={a[r]}"; r -= 1
        else:
            act = f"< {k}: bỏ a[l]={a[l]}"; l += 1
        print(f"  tổng={s:>2} {act:<22} -> l={l}, r={r}, ops={ops}")
    return ops

trace_two_pointers([3, 1, 3, 4, 3], 6)
trace_two_pointers([1, 2, 3, 4], 5)

sắp xếp: [1, 3, 3, 3, 4]
  tổng= 5 < 6: bỏ a[l]=1         -> l=1, r=4, ops=0
  tổng= 7 > 6: bỏ a[r]=4         -> l=1, r=3, ops=0
  tổng= 6 == 6: ghép (3, 3)      -> l=2, r=2, ops=1
sắp xếp: [1, 2, 3, 4]
  tổng= 5 == 5: ghép (1, 4)      -> l=1, r=2, ops=1
  tổng= 5 == 5: ghép (2, 3)      -> l=2, r=1, ops=2


2

### 8.2 Trace cách 3 (bảng băm) trên ví dụ tự dựng `[1,1,1,5,5,2,4,4,4]`, `k = 6`

Ví dụ tự dựng — đáp án `3` được kiểm chứng bằng vét cạn ở mục 12 (đối chiếu lại đề gốc nếu muốn).

In [13]:
def trace_hash(nums, k):
    waiting = Counter(); ops = 0
    for x in nums:
        if waiting[k - x] > 0:
            waiting[k - x] -= 1; ops += 1
            print(f"  x={x}: có {k - x} đang chờ -> ghép, ops={ops}, chờ={dict(+waiting)}")
        else:
            waiting[x] += 1
            print(f"  x={x}: chưa có {k - x} -> cho {x} vào chờ, chờ={dict(+waiting)}")
    return ops

trace_hash([1, 1, 1, 5, 5, 2, 4, 4, 4], 6)

  x=1: chưa có 5 -> cho 1 vào chờ, chờ={1: 1}
  x=1: chưa có 5 -> cho 1 vào chờ, chờ={1: 2}
  x=1: chưa có 5 -> cho 1 vào chờ, chờ={1: 3}
  x=5: có 1 đang chờ -> ghép, ops=1, chờ={1: 2}
  x=5: có 1 đang chờ -> ghép, ops=2, chờ={1: 1}
  x=2: chưa có 4 -> cho 2 vào chờ, chờ={1: 1, 2: 1}
  x=4: có 2 đang chờ -> ghép, ops=3, chờ={1: 1}
  x=4: chưa có 2 -> cho 4 vào chờ, chờ={1: 1, 4: 1}
  x=4: chưa có 2 -> cho 4 vào chờ, chờ={1: 1, 4: 2}


3

(`+waiting` bỏ các khóa có giá trị 0 khi in.)

## 9. Code nộp LeetCode

In [14]:
class Solution:
    def maxOperations(self, nums: List[int], k: int) -> int:
        waiting = Counter()                 # các số đang chờ được ghép
        ops = 0
        for x in nums:
            if waiting[k - x]:
                waiting[k - x] -= 1
                ops += 1
            else:
                waiting[x] += 1
        return ops

for nums_, k_, exp in examples:
    assert Solution().maxOperations(nums_, k_) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Chặn trên.** Mọi thao tác dùng một cặp `{x, k − x}`. Với `x < k − x`, mỗi thao tác của nhóm tiêu một `x` và một `k − x` ⇒ không quá `min(cnt[x], cnt[k−x])` thao tác. Với `x = k − x`: mỗi thao tác tiêu hai `x` ⇒ không quá `cnt[x] // 2`. Các nhóm rời nhau ⇒ đáp án `≤ Σ` (công thức cách 4).

**Đạt được.** Ghép tùy ý trong mỗi nhóm đạt đúng chặn trên ⇒ công thức là đáp án.

**Cách 3 (băm) đạt công thức.** Xét một nhóm `{x, y}` với `x ≠ y`. Bất biến: sau mỗi bước, trong hàng chờ **không** đồng thời có `x` và `y` (vì khi `x` tới mà có `y` chờ thì ghép ngay). Khi duyệt xong, số phần tử còn chờ của nhóm là `|cnt[x] − cnt[y]|`, số cặp đã ghép là `(cnt[x] + cnt[y] − |cnt[x] − cnt[y]|) / 2 = min(cnt[x], cnt[y])`. Với `x = y`: các bản sao lần lượt "chờ – ghép – chờ – ghép" ⇒ `cnt[x] // 2` cặp.

**Cách 2 (hai con trỏ).** Khi `a[l] + a[r] > k`, `a[r]` không ghép được với bất kỳ phần tử nào trong `a[l..r−1]` (đều `≥ a[l]`) nên bỏ `a[r]` không mất cặp nào; tương tự khi `< k`. Khi `== k`: tồn tại lời giải tối ưu ghép `a[l]` với `a[r]` (lập luận hoán đổi: nếu tối ưu ghép `a[l]` với `a[j]` và `a[r]` với `a[i]`, thì `a[j] = a[r]` về giá trị — đổi chỗ không đổi số cặp). Vậy bất biến ở mục 6 được giữ.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1a. Vét cạn ghép | hàm mũ | hàm mũ |
| 1b. Tham lam `O(n²)` | `O(n²)` | `O(n)` |
| 2. Sắp xếp + hai con trỏ | `O(n log n)` | `O(n)` (hoặc `O(1)` nếu sắp tại chỗ) |
| 3. Bảng băm một lượt | `O(n)` trung bình | `O(n)` |
| 4. Công thức `Counter` | `O(n)` trung bình | `O(n)` |

In [15]:
rnd = random.Random(1679)
big = [rnd.randint(1, 10**6) for _ in range(10**5)]
kb = 10**6
for name, fn in [("two_ptr", max_ops_two_pointers), ("hash", max_ops_hash), ("counter", max_ops_counter), ("Solution", Solution().maxOperations)]:
    t0 = time.perf_counter(); r = fn(big, kb); dt = time.perf_counter() - t0
    print(f"{name:<9}: {r:>6}  {dt * 1000:6.1f} ms")
t0 = time.perf_counter(); r = max_ops_quadratic(big[:3000], kb); dt = time.perf_counter() - t0
print(f"quadratic n=3000: {r} ({dt * 1000:.0f} ms) | hash: {max_ops_hash(big[:3000], kb)}")

two_ptr  :   4677    23.7 ms
hash     :   4677    28.4 ms
counter  :   4677    16.2 ms
Solution :   4677    23.2 ms


quadratic n=3000: 4 (167 ms) | hash: 4


## 12. Bộ test

In [16]:
ALL = {
    "quadratic": max_ops_quadratic,
    "two_ptr": max_ops_two_pointers,
    "hash": max_ops_hash,
    "counter": max_ops_counter,
    "SolTwo": lambda a, k: SolutionTwoPointers().maxOperations(list(a), k),
    "Solution": lambda a, k: Solution().maxOperations(list(a), k),
}

def check(nums, k, exp=None, brute=True):
    ref = max_ops_definition(nums, k) if brute else max_ops_counter(nums, k)
    for name, fn in ALL.items():
        if not brute and name == "quadratic":
            continue
        got = fn(list(nums), k)
        assert got == ref, (name, nums, k, got, ref)
    if exp is not None:
        assert ref == exp, (nums, k, ref, exp)
    return ref

for nums_, k_, exp in examples:
    check(nums_, k_, exp)
print("Ví dụ đề: OK")

check([1], 2, 0)
check([1, 1], 2, 1)
check([1, 1, 1], 2, 1)
check([3, 3, 3, 3], 6, 2)
check([1, 1, 1, 5, 5, 2, 4, 4, 4], 6, 3)      # ví dụ tự dựng ở mục 4/8
check([5, 5], 5, 0)
check([2, 2, 2, 3, 1, 1, 4, 1], 4, 2)          # tự dựng: nhóm {2}: 3//2=1, nhóm {3,1}: min(1,3)=1
print("Edge case: OK")

Ví dụ đề: OK
Edge case: OK


In [17]:
rnd = random.Random(2013)
for _ in range(3000):
    n = rnd.randint(1, 10)
    nums_ = [rnd.randint(1, 8) for _ in range(n)]
    k_ = rnd.randint(1, 16)
    check(nums_, k_)
print("3000 mảng ngẫu nhiên nhỏ so với vét cạn ghép cặp: OK")

for _ in range(50):
    n = rnd.randint(1, 5000)
    nums_ = [rnd.randint(1, 50) for _ in range(n)]
    check(nums_, rnd.randint(1, 100), brute=False)
print("50 mảng cỡ vừa so giữa các cách nhanh: OK")

3000 mảng ngẫu nhiên nhỏ so với vét cạn ghép cặp: OK
50 mảng cỡ vừa so giữa các cách nhanh: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Đếm số **cặp chỉ số** `i < j` có tổng `k`

In [18]:
def max_ops_bug_pairs(nums, k):
    n = len(nums)
    return sum(nums[i] + nums[j] == k for i in range(n) for j in range(i + 1, n))   # SAI

for nums_, k_ in [([3, 1, 3, 4, 3], 6), ([3, 3, 3, 3], 6)]:
    print(f"{nums_}, k={k_}: sai = {max_ops_bug_pairs(nums_, k_)}, đúng = {Solution().maxOperations(nums_, k_)}")

[3, 1, 3, 4, 3], k=6: sai = 3, đúng = 1
[3, 3, 3, 3], k=6: sai = 6, đúng = 2


**Vì sao sai:** mỗi thao tác **xóa** hai số; một số không thể nằm trong hai thao tác. `[3,3,3,3]` có 6 cặp chỉ số nhưng chỉ 2 thao tác. **Sửa:** ghép rời nhau (băm / hai con trỏ / công thức).

### Lỗi 2 — Công thức đếm cả hai chiều `x` và `k − x`

In [19]:
def max_ops_bug_double(nums, k):
    c = Counter(nums)
    return sum(min(cx, c.get(k - x, 0)) if x != k - x else cx // 2 for x, cx in c.items())   # SAI: nhóm {x, k-x} tính 2 lần

nums_, k_ = [1, 2, 3, 4], 5
print("sai :", max_ops_bug_double(nums_, k_), "| đúng:", Solution().maxOperations(nums_, k_))

sai : 4 | đúng: 2


**Vì sao sai:** nhóm `{1, 4}` được cộng khi `x = 1` **và** khi `x = 4`. **Sửa:** chỉ cộng khi `x < k − x`, xử lý riêng `x == k − x`.

### Lỗi 3 — Quên trường hợp `x == k − x` (ghép một số với **chính nó**)

In [20]:
def max_ops_bug_self(nums, k):
    c = Counter(nums)
    ops = 0
    for x, cx in c.items():
        y = k - x
        if x < y:
            ops += min(cx, c.get(y, 0))
        elif x == y:
            ops += cx                       # SAI: một phần tử không thể ghép với chính nó
    return ops

nums_, k_ = [3, 1, 3, 4, 3], 6
print("sai :", max_ops_bug_self(nums_, k_), "| đúng:", Solution().maxOperations(nums_, k_))

sai : 3 | đúng: 1


**Vì sao sai:** cặp `{3, 3}` cần **hai** phần tử giá trị 3. Ba số 3 chỉ cho 1 cặp. **Sửa:** `cx // 2`. Tương tự trong bản băm, phải kiểm tra "có `k − x` **khác** đang chờ" trước khi tự thêm `x` vào hàng chờ — nếu thêm `x` trước rồi mới kiểm, `x` sẽ tự ghép với chính nó.

In [21]:
def max_ops_bug_hash_order(nums, k):
    waiting = Counter(); ops = 0
    for x in nums:
        waiting[x] += 1                     # SAI: thêm trước khi kiểm tra
        if waiting[k - x] > 0:
            waiting[k - x] -= 1
            ops += 1
    return ops

nums_, k_ = [3, 1, 4], 6
print("thêm trước khi kiểm:", max_ops_bug_hash_order(nums_, k_), "| đúng:", Solution().maxOperations(nums_, k_))

thêm trước khi kiểm: 1 | đúng: 0


### Lỗi 4 — Hai con trỏ trên mảng **chưa sắp xếp**

In [22]:
def max_ops_bug_unsorted(nums, k):
    a = list(nums)                          # SAI: quên sắp xếp
    l, r, ops = 0, len(a) - 1, 0
    while l < r:
        s = a[l] + a[r]
        if s == k:
            ops += 1; l += 1; r -= 1
        elif s > k:
            r -= 1
        else:
            l += 1
    return ops

nums_, k_ = [4, 1, 3, 2], 5
print("chưa sắp:", max_ops_bug_unsorted(nums_, k_), "| đúng:", Solution().maxOperations(nums_, k_))
nums_, k_ = [2, 5, 4, 3, 1], 6
print("chưa sắp:", max_ops_bug_unsorted(nums_, k_), "| đúng:", Solution().maxOperations(nums_, k_))

chưa sắp: 1 | đúng: 2
chưa sắp: 1 | đúng: 2


**Vì sao sai:** lập luận "bỏ `a[r]` vì quá lớn với mọi phần tử còn lại" chỉ đúng khi mảng **tăng dần**. **Sửa:** `a = sorted(nums)`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Cặp có tổng `k`" ⇒ nghĩ ngay tới **Two Sum**: bảng băm tra phần bù, hoặc sắp xếp + hai con trỏ.
- "Mỗi phần tử dùng **tối đa một lần**" ⇒ ghép rời nhau; bảng băm phải **giảm** số đếm khi ghép.
- Bài ghép cặp theo **giá trị** ⇒ thường có **công thức theo nhóm** `min(cnt[x], cnt[k−x])`.
- Trường hợp đặc biệt `x == k − x` luôn cần xử lý riêng.
- Nếu đề cho phép sửa mảng và muốn `O(1)` bộ nhớ ⇒ sắp tại chỗ + hai con trỏ.

## 15. Biến thể của bài

1. **LeetCode 1 — Two Sum:** tìm **một** cặp chỉ số.
2. **Đếm số cặp chỉ số `i < j`** có tổng `k` (không xóa): `Σ cnt[x]·cnt[k−x]` cho `x < k−x` + `C(cnt[k/2], 2)`.
3. **LeetCode 1010 — Pairs of Songs With Total Durations Divisible by 60:** tổng **chia hết** cho `k` ⇒ làm việc với số dư.
4. **LeetCode 1497 — Check If Array Pairs Are Divisible by k:** có ghép **hết** được không.
5. **Ghép cặp có tổng `≤ k`, tối đa số cặp** (LeetCode 881 — Boats to Save People): sắp xếp + hai con trỏ.
6. **Ba số có tổng `k`** (3Sum): cố định một số, hai con trỏ cho phần còn lại.

In [23]:
def count_index_pairs(nums, k):
    c = Counter(nums)
    total = 0
    for x, cx in c.items():
        y = k - x
        if x < y:
            total += cx * c.get(y, 0)
        elif x == y:
            total += cx * (cx - 1) // 2
    return total

def can_pair_all_divisible(arr, k):
    """LeetCode 1497."""
    c = Counter(x % k for x in arr)
    if c[0] % 2:
        return False
    for r in range(1, k):
        if r == k - r:
            if c[r] % 2: return False
        elif c[r] != c[k - r]:
            return False
    return True

print("đếm cặp chỉ số [3,1,3,4,3], k=6:", count_index_pairs([3, 1, 3, 4, 3], 6), "| kiểm:", max_ops_bug_pairs([3, 1, 3, 4, 3], 6))
print("1497 [1,2,3,4,5,10,6,7,8,9], k=5:", can_pair_all_divisible([1, 2, 3, 4, 5, 10, 6, 7, 8, 9], 5))

đếm cặp chỉ số [3,1,3,4,3], k=6: 3 | kiểm: 3
1497 [1,2,3,4,5,10,6,7,8,9], k=5: True


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1a. Vét cạn ghép | thử ghép/bỏ, nhớ đa tập | hàm mũ | hàm mũ | chuẩn đối chiếu |
| 1b. Tham lam `O(n²)` | tìm bạn ghép bằng quét | `O(n²)` | `O(n)` | hiểu bài |
| 2. Hai con trỏ | sắp xếp, bỏ đầu không ghép được | `O(n log n)` | `O(1)`–`O(n)` | khi cần ít bộ nhớ |
| 3. Băm một lượt | hàng chờ phần bù | `O(n)` | `O(n)` | **nộp bài**, streaming |
| 4. Công thức | `Σ min(cnt[x], cnt[k−x])` | `O(n)` | `O(n)` | muốn biết cấu trúc đáp án |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `Solution().maxOperations([2, 2, 2, 2, 2], 4)` và `Solution().maxOperations([1, 2, 3, 4, 5], 10)`?

**Bài 2 (tính tay):** Dùng **công thức theo nhóm** (không chạy code) để tính đáp án cho `nums = [4, 4, 1, 3, 1, 3, 2, 2, 5, 5, 1, 5, 2, 1, 2, 3, 5, 4]` với `k = 2`, và cho `nums = [2, 5, 4, 4, 1, 3, 4, 4, 1, 4, 4, 1, 2, 1, 2, 2, 3, 2, 4, 2]` với `k = 3`.

**Bài 3 (điền TODO):** Hoàn thiện `list_operations(nums, k)` trả về **danh sách cặp chỉ số** (theo mảng gốc) của một phương án tối ưu.

**Bài 4 (tự viết lại):** Viết lại cách 2 **sắp xếp tại chỗ** (dùng `nums.sort()`) và đếm luôn số lần mỗi nhánh (`==`, `>`, `<`) được thực hiện.

**Bài 5 (mini challenge):** Biến thể "tổng chia hết cho `k`": viết `max_ops_divisible(nums, k)` = số thao tác tối đa, mỗi thao tác xóa hai số có tổng **chia hết** cho `k`. Kiểm bằng vét cạn.

In [24]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def list_operations(nums, k):
    # TODO: như bản băm nhưng hàng chờ lưu danh sách CHỈ SỐ theo giá trị
    pass

# Bài 4 — TODO
def two_pointers_inplace_stats(nums, k):
    # TODO: nums.sort(); đếm số lần ==, >, <; trả (ops, stats)
    pass

# Bài 5 — TODO
def max_ops_divisible(nums, k):
    # TODO: nhóm theo số dư r = x % k; ghép r với (k - r) % k; r == 0 và 2r == k ghép nội bộ
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: Năm số 2 với `k = 4`: nhóm `{2}`.
- Mức 2: `5 // 2 = 2`.
- Mức 3: `[1..5]`, `k = 10`: chỉ `5 + 5` — nhưng chỉ có một số 5.

**Bài 2**
- Mức 1: Với `k = 2`, chỉ có cặp `{1, 1}`.
- Mức 2: Đếm số lần xuất hiện của 1 rồi chia đôi.
- Mức 3: Với `k = 3`, nhóm `{1, 2}`: `min(cnt[1], cnt[2])`.

**Bài 3**
- Mức 1: `waiting[value] = [các chỉ số đang chờ]`.
- Mức 2: Ghép thì `pop()` một chỉ số của `k − x`.
- Mức 3: Kiểm bằng `simulate` và so số cặp với `Solution`.

**Bài 4**
- Mức 1: `list.sort()` sắp tại chỗ, trả `None`.
- Mức 2: `stats = {'eq': 0, 'gt': 0, 'lt': 0}`.
- Mức 3: Tổng số bước ≤ `n`.

**Bài 5**
- Mức 1: Chỉ số dư `x % k` quan trọng.
- Mức 2: Nhóm `{r, k − r}` với `0 < r < k − r`: `min(cnt[r], cnt[k − r])`.
- Mức 3: `r = 0` và (nếu `k` chẵn) `r = k/2`: `cnt // 2`.

## 19. Đáp án tham khảo cho bài tập

In [25]:
# Bài 1
r1 = (Solution().maxOperations([2, 2, 2, 2, 2], 4), Solution().maxOperations([1, 2, 3, 4, 5], 10))
print("Bài 1:", r1)
assert r1 == (2, 0)

# Bài 2
a2 = [4, 4, 1, 3, 1, 3, 2, 2, 5, 5, 1, 5, 2, 1, 2, 3, 5, 4]
b2 = [2, 5, 4, 4, 1, 3, 4, 4, 1, 4, 4, 1, 2, 1, 2, 2, 3, 2, 4, 2]
print("Bài 2: k=2 ->", Solution().maxOperations(a2, 2), "(số 1:", a2.count(1), ")")
print("       k=3 ->", Solution().maxOperations(b2, 3), "(số 1:", b2.count(1), ", số 2:", b2.count(2), ")")
assert Solution().maxOperations(a2, 2) == a2.count(1) // 2
assert Solution().maxOperations(b2, 3) == min(b2.count(1), b2.count(2))

# Bài 3
def list_operations_sol(nums, k):
    waiting = {}
    plan = []
    for i, x in enumerate(nums):
        lst = waiting.get(k - x)
        if lst:
            plan.append((lst.pop(), i))
        else:
            waiting.setdefault(x, []).append(i)
    return plan

rnd = random.Random(3)
for _ in range(2000):
    nums_ = [rnd.randint(1, 8) for _ in range(rnd.randint(1, 15))]
    k_ = rnd.randint(1, 16)
    plan = list_operations_sol(nums_, k_)
    assert simulate(nums_, k_, plan) == Solution().maxOperations(nums_, k_)
print("Bài 3 OK | ví dụ 1:", list_operations_sol([1, 2, 3, 4], 5))

Bài 1: (2, 0)
Bài 2: k=2 -> 2 (số 1: 4 )
       k=3 -> 4 (số 1: 4 , số 2: 6 )
Bài 3 OK | ví dụ 1: [(1, 2), (0, 3)]


In [26]:
# Bài 4
def two_pointers_inplace_stats_sol(nums, k):
    nums.sort()
    stats = {'eq': 0, 'gt': 0, 'lt': 0}
    l, r, ops = 0, len(nums) - 1, 0
    while l < r:
        s = nums[l] + nums[r]
        if s == k:
            stats['eq'] += 1; ops += 1; l += 1; r -= 1
        elif s > k:
            stats['gt'] += 1; r -= 1
        else:
            stats['lt'] += 1; l += 1
    return ops, stats

arr = [3, 1, 3, 4, 3]
print("Bài 4:", two_pointers_inplace_stats_sol(arr, 6), "| mảng sau khi sắp tại chỗ:", arr)
rnd = random.Random(4)
for _ in range(1000):
    nums_ = [rnd.randint(1, 20) for _ in range(rnd.randint(1, 30))]
    k_ = rnd.randint(1, 40)
    ops, st = two_pointers_inplace_stats_sol(list(nums_), k_)
    assert ops == Solution().maxOperations(nums_, k_) and 2 * st['eq'] + st['gt'] + st['lt'] <= len(nums_)

# Bài 5
def max_ops_divisible_sol(nums, k):
    c = Counter(x % k for x in nums)
    ops = c[0] // 2
    for r in range(1, k):
        s = k - r
        if r < s:
            ops += min(c[r], c[s])
        elif r == s:
            ops += c[r] // 2
    return ops

def max_ops_divisible_brute(nums, k):
    @lru_cache(maxsize=None)
    def best(t):
        if len(t) < 2:
            return 0
        x, rest = t[0], t[1:]
        res = best(rest)
        for i, y in enumerate(rest):
            if (x + y) % k == 0:
                res = max(res, 1 + best(rest[:i] + rest[i + 1:]))
        return res
    return best(tuple(sorted(nums)))

rnd = random.Random(5)
for _ in range(800):
    nums_ = [rnd.randint(1, 12) for _ in range(rnd.randint(1, 9))]
    k_ = rnd.randint(1, 7)
    assert max_ops_divisible_sol(nums_, k_) == max_ops_divisible_brute(nums_, k_), (nums_, k_)
print("Bài 4, Bài 5 OK | divisible([1,2,3,4,5,10,6,7,8,9], 5) =", max_ops_divisible_sol([1, 2, 3, 4, 5, 10, 6, 7, 8, 9], 5))

Bài 4: (1, {'eq': 1, 'gt': 1, 'lt': 1}) | mảng sau khi sắp tại chỗ: [1, 3, 3, 3, 4]
Bài 4, Bài 5 OK | divisible([1,2,3,4,5,10,6,7,8,9], 5) = 5


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "ghép cặp rời nhau có tổng k" (băm một lượt):**

```python
waiting = Counter()
ops = 0
for x in nums:
    if waiting[k - x]:          # KIỂM TRA trước
        waiting[k - x] -= 1
        ops += 1
    else:
        waiting[x] += 1         # rồi mới cho vào hàng chờ
return ops
```

**Template hai con trỏ:** `a.sort(); l, r = 0, n − 1; ==: ghép, l++, r−−; >: r−−; <: l++`.

**Template công thức:** `Σ_{x < k−x} min(cnt[x], cnt[k−x]) + cnt[k/2] // 2`.

**Checklist:**
- [ ] Ghép **rời nhau** (không đếm cặp chỉ số)?
- [ ] Xử lý `x == k − x` bằng `// 2`?
- [ ] Bản băm: kiểm tra phần bù **trước** khi thêm `x`?
- [ ] Bản hai con trỏ: đã **sắp xếp**?
- [ ] Công thức: mỗi nhóm `{x, k − x}` chỉ cộng **một lần**?
- [ ] Đã so với vét cạn ghép cặp trên mảng nhỏ?
- [ ] Chữ ký đúng: `def maxOperations(self, nums: List[int], k: int) -> int`?